In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib import cm
from matplotlib.colors import Normalize
from collections import defaultdict

import json
import yaml
import h5py
import tqdm
# from larndsim.fee import digitize


In [2]:
def unique_channel_id(d):
    return ((d['io_group'].astype(int)*1000+d['io_channel'].astype(int))*1000 \
            + d['chip_id'].astype(int))*100 + d['channel_id'].astype(int)

def unique_to_channel_id(unique):
    return unique % 100

def unique_to_chip_id(unique):
    return (unique// 100) % 1000

def unique_to_io_channel(unique):
    return(unique//(100*1000)) % 1000

def unique_to_tiles(unique):
    return ( (unique_to_io_channel(unique)-1) // 4) + 1

def unique_to_io_group(unique):
    return(unique // (100*1000*1000)) % 1000



In [13]:
_default_geometry_yaml = '../larndsim/pixel_layouts/multi_tile_layout-3.0.40_NDLArModule_v3.yaml'

def _default_pxy():
    return (0., 0.)


def _rotate_pixel(pixel_pos, tile_orientation):
    return pixel_pos[0]*tile_orientation[2], pixel_pos[1]*tile_orientation[1]


cmap = cm.viridis_r
pixel_pitch = 1

geometry_yaml = _default_geometry_yaml
with open(geometry_yaml) as fi:
    geo = yaml.full_load(fi)

pixel_pitch = geo['pixel_pitch']

chip_channel_to_position = geo['chip_channel_to_position']
tile_orientations = geo['tile_orientations']
tile_positions = geo['tile_positions']
tile_indeces = geo['tile_indeces']
xs = np.array(list(chip_channel_to_position.values()))[
    :, 0] * pixel_pitch
ys = np.array(list(chip_channel_to_position.values()))[
    :, 1] * pixel_pitch
x_size = max(xs)-min(xs)+pixel_pitch
y_size = max(ys)-min(ys)+pixel_pitch

tile_geometry = defaultdict(int)
io_group_io_channel_to_tile = {}
geometry = defaultdict(_default_pxy)

for tile in geo['tile_chip_to_io']:
    tile_orientation = tile_orientations[tile]
    tile_geometry[tile] = tile_positions[tile], tile_orientations[tile]
    for chip in geo['tile_chip_to_io'][tile]:
        io_group_io_channel = geo['tile_chip_to_io'][tile][chip]
        io_group = io_group_io_channel//1000
        io_channel = io_group_io_channel % 1000
        io_group_io_channel_to_tile[(
            io_group, io_channel)] = tile

    for chip_channel in geo['chip_channel_to_position']:
        chip = chip_channel // 1000
        channel = chip_channel % 1000
        try:
            io_group_io_channel = geo['tile_chip_to_io'][tile][chip]
        except KeyError:
            print("Chip %i on tile %i not present in network" %
                  (chip, tile))
            continue

        io_group = io_group_io_channel // 1000
        io_channel = io_group_io_channel % 1000
        x = chip_channel_to_position[chip_channel][0] * \
            pixel_pitch + pixel_pitch / 2 - x_size / 2
        y = chip_channel_to_position[chip_channel][1] * \
            pixel_pitch + pixel_pitch / 2 - y_size / 2

        x, y = _rotate_pixel((x, y), tile_orientation)
        x += tile_positions[tile][2] 
        y += tile_positions[tile][1] 

        geometry[(io_group, io_group_io_channel_to_tile[(
            io_group, io_channel)], chip, channel)] = x, y

xmin = min(np.array(list(geometry.values()))[:, 0])-pixel_pitch/2
xmax = max(np.array(list(geometry.values()))[:, 0])+pixel_pitch/2
ymin = min(np.array(list(geometry.values()))[:, 1])-pixel_pitch/2
ymax = max(np.array(list(geometry.values()))[:, 1])+pixel_pitch/2

tile_vertical_lines = np.linspace(xmin, xmax, 3)
tile_horizontal_lines = np.linspace(ymin, ymax, 5)
chip_vertical_lines = np.linspace(xmin, xmax, 21)
chip_horizontal_lines = np.linspace(ymin, ymax, 41)



In [18]:
def unique_id_to_pixel_id(un):
    io_group = unique_to_io_group(un)
    tile = unique_to_io_channel(un)
    chip_id = unique_to_chip_id(un)
    channel_id = unique_to_channel_id(un)

    pitch = pixel_pitch

    x, y = geometry[(io_group, tile + 10 * (io_group - 1), chip_id, channel_id)]
    if (x==0. and y==0.):
        print((io_group, tile + 10 * (io_group - 1), chip_id, channel_id))
    

    x_min = xmin
    x_max = xmax
    
    y_min = ymin
    y_max = ymax

    x_int = (x - x_min) / pitch - 0.5
    y_int = (y - y_min) / pitch - 0.5

    if abs(round(x_int) - x_int) > 0.05:
        print('ERROR X: ', un, ' - ', round(x_int), ' vs ', x_int)
    if abs(round(y_int) - y_int) > 0.05:
        print('ERROR Y: ', un, ' - ', round(y_int), ' vs ', y_int)
        
    if abs(round((x_max - x_min)/pitch) - (x_max - x_min)/pitch) > 0.05:
        print('ERROR STEP X: ', un)
    if abs(round((y_max - y_min)/pitch) - (y_max - y_min)/pitch) > 0.05:
        print('ERROR STEP Y: ', un)

    npix_x = round((x_max - x_min)/pitch)
    npix_y = round((y_max - y_min)/pitch)
    
    return round(x_int) + (round(y_int) + npix_y * ((((io_group-1)//2) % 2))) * npix_x
    

In [19]:
pixelid_to_uniqueid = dict()
uniqueid_to_pixelid = dict()

for io_group in tqdm.tqdm(range(1, 5)):
    for tile in range(1, 11):
        for chip_id in range(11, 171):
            for channel_id in range(64):

                unique_id = ((io_group*1000+tile)*1000 + chip_id)*100 + channel_id
                pixel_id = unique_id_to_pixel_id(unique_id)

                if pixel_id in pixelid_to_uniqueid.keys():
                    print('DUPLICATE PIXEL ID')
                    print('pixel_id: ', pixel_id)
                    print('channel: ', (io_group, tile, chip_id, channel_id))
                if channel_id in uniqueid_to_pixelid.keys():
                    print('DUPLICATE UNIQUE ID')
                pixelid_to_uniqueid[pixel_id] = unique_id
                uniqueid_to_pixelid[unique_id] = pixel_id

with open("pixelid_to_uniqueid_fsd.json", "w") as f:
    json.dump(pixelid_to_uniqueid, f)
with open("uniqueid_to_pixelid_fsd.json", "w") as f:
    json.dump(uniqueid_to_pixelid, f)


100%|██████████| 4/4 [00:04<00:00,  1.21s/it]


In [8]:
geometry[(1, 1, 11, 0)]

(-1.8600000000000136, 1471.26)

In [7]:
geometry[(3, 21, 11, 0)]

(-474.3, 1471.26)

In [21]:
for k in geometry.keys():
    if k[0] == 4:
        print('Here')
        print(k)
        break

Here
(4, 31, 11, 0)


In [22]:
pixelid_to_uniqueid[220215]

301007332

In [10]:
64*10*16*30

307200

In [17]:
unique_id_to_pixel_id(301002021)

5

**
238389
3-35-74
20
**
220215
3-39-73
32